In [5]:
from models import SEASModel
from evaluation_utils import evaluate_iplg_convergence
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset
from torch.utils.data import DataLoader
import torch
from torch.nn import CrossEntropyLoss
import os
import pickle
from train_utils import train_IPLG
from data_utils import latent_MH_collate_fn
from pprint import pprint
import pandas as pd

device_name = 'cuda:0'
batch_size = 4

In [6]:
if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

In [7]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)
mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

In [8]:
loss_scheme = 'l'

d_model = 512
transformer_model = SEASModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=d_model,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=d_model,
    device=device,
)
checkpoint = torch.load(f'saved_models/iplg/SE/iplg_{loss_scheme}_loss.pt', map_location=device_name)
transformer_model.load_state_dict(checkpoint)
transformer_model.to(device)
transformer_model.eval()

SEASModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithFiLM(
      (layer): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
      (film): FiLMAdapter(
        (gamma): Linear(in_features=512, out_features=512, bias=True)
        (beta): Linea

In [9]:
jazz_folder = 'MIDIs/jazz/real'
nott_folder = 'MIDIs/nottingham/real'

jazz_piece_path = os.path.join(jazz_folder, 'real_0.mid')
nott_piece_path = os.path.join(nott_folder, 'real_0.mid')

In [10]:
jazz_encoded = tokenizer.encode(jazz_piece_path)
nott_encoded = tokenizer.encode(nott_piece_path)

In [11]:
print(jazz_encoded.keys())

dict_keys(['harmony_tokens', 'harmony_ids', 'pianoroll', 'time_signature', 'attention_mask', 'skip_steps', 'melody_part', 'ql_per_quantum', 'back_interval', 'harmonic_rhythm_density', 'harmonic_complexity', 'h_density_complexity'])


In [16]:
melody_grid = torch.tensor(jazz_encoded['pianoroll']).unsqueeze(0).to(device)
harmony_ids = torch.tensor(jazz_encoded['harmony_ids']).unsqueeze(0).to(device)

In [17]:
y, layers_output = transformer_model(melody_grid, harmony_ids, get_layers_output=True)

In [19]:
print(layers_output[0].shape)

torch.Size([1, 160, 512])
